In [2]:

from pathlib import Path
import hashlib
import csv
import zipfile
from datetime import datetime
import shutil

# Purpose: Package SWaT data and frozen NB08/NB09 artifacts
# for transfer to Kaggle, preserving the original project files.

PROJECT_ROOT = Path(r"C:\Users\ADEGOKE\Desktop\SWATS")

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}\n"
        "Confirm that this notebook is running on your Windows computer."
    )

# Required directories and data files.
required_dirs = [
    PROJECT_ROOT / "nb08_artifacts",
    PROJECT_ROOT / "nb09_artifacts",
    PROJECT_ROOT / "data" / "raw",
]

required_files = [
    PROJECT_ROOT / "data" / "raw" / "normal.csv",
    PROJECT_ROOT / "data" / "raw" / "attack.csv",
]

for path in required_dirs:
    if not path.is_dir():
        raise FileNotFoundError(f"Required directory missing: {path}")

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(f"Required data file missing: {path}")

for folder in required_dirs[:2]:
    if not any(p.is_file() for p in folder.rglob("*")):
        raise FileNotFoundError(
            f"Frozen artifact directory is empty: {folder}"
        )

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
package_name = f"SWATS_Kaggle_Transfer_{timestamp}"
staging_dir = PROJECT_ROOT / f"_temporary_{package_name}"
zip_path = PROJECT_ROOT / f"{package_name}.zip"

if staging_dir.exists() or zip_path.exists():
    raise FileExistsError(
        f"Package destination already exists:\n"
        f"{staging_dir}\n{zip_path}"
    )

try:
    staging_dir.mkdir()

    # Copy frozen artifacts in full.
    for folder_name in ("nb08_artifacts", "nb09_artifacts"):
        shutil.copytree(
            PROJECT_ROOT / folder_name,
            staging_dir / folder_name
        )

    # Include the two separate raw datasets.
    raw_destination = staging_dir / "data" / "raw"
    raw_destination.mkdir(parents=True)

    for source in required_files:
        shutil.copy2(source, raw_destination / source.name)

    # Include the original checksum record if present.
    checksum_file = PROJECT_ROOT / "_nb08_checksums_BASELINE.csv"
    if checksum_file.is_file():
        shutil.copy2(checksum_file, staging_dir / checksum_file.name)

    # Include common project metadata and dependency files if present.
    optional_files = [
        "requirements.txt",
        "pyproject.toml",
        "environment.yml",
        "environment.yaml",
        "uv.lock",
        "README.md",
        "PROJECT_STATE.md",
    ]

    for name in optional_files:
        source = PROJECT_ROOT / name
        if source.is_file():
            shutil.copy2(source, staging_dir / name)

    # Include common source-code directories when they exist.
    # Notebooks and the existing NB10 output directory are excluded.
    optional_dirs = ["src", "scripts", "models", "utils"]

    for name in optional_dirs:
        source = PROJECT_ROOT / name
        if source.is_dir():
            shutil.copytree(source, staging_dir / name)

    # Hash every staged file and record its relative path and size.
    manifest_path = staging_dir / "TRANSFER_MANIFEST.csv"

    with manifest_path.open(
        "w", newline="", encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=["relative_path", "size_bytes", "sha256"]
        )
        writer.writeheader()

        for path in sorted(staging_dir.rglob("*")):
            if not path.is_file() or path == manifest_path:
                continue

            digest = hashlib.sha256()
            with path.open("rb") as source_file:
                for chunk in iter(
                    lambda: source_file.read(8 * 1024 * 1024), b""
                ):
                    digest.update(chunk)

            writer.writerow({
                "relative_path": path.relative_to(staging_dir).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": digest.hexdigest(),
            })

    # Build the ZIP.
    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED
    ) as archive:
        for path in sorted(staging_dir.rglob("*")):
            if path.is_file():
                archive.write(
                    path,
                    arcname=path.relative_to(staging_dir).as_posix()
                )

    # Verify ZIP integrity and required entries.
    with zipfile.ZipFile(zip_path, "r") as archive:
        bad_entry = archive.testzip()
        if bad_entry is not None:
            raise RuntimeError(f"ZIP integrity check failed: {bad_entry}")

        names = set(archive.namelist())
        required_entries = {
            "nb08_artifacts/",
            "nb09_artifacts/",
            "data/raw/normal.csv",
            "data/raw/attack.csv",
            "TRANSFER_MANIFEST.csv",
        }

        # ZIPs may omit explicit directory entries, so check file prefixes.
        for folder_name in ("nb08_artifacts/", "nb09_artifacts/"):
            if not any(name.startswith(folder_name) for name in names):
                raise RuntimeError(
                    f"ZIP is missing artifact files under {folder_name}"
                )

        for entry in (
            "data/raw/normal.csv",
            "data/raw/attack.csv",
            "TRANSFER_MANIFEST.csv",
        ):
            if entry not in names:
                raise RuntimeError(f"ZIP is missing required entry: {entry}")

    print("PACKAGE CREATED AND ZIP INTEGRITY VERIFIED")
    print(f"Location: {zip_path}")
    print(f"Size: {zip_path.stat().st_size / (1024**2):,.2f} MB")
    print(f"Files packaged: {len(names)}")
    print("Original project files were not modified.")

finally:
    if staging_dir.exists():
        shutil.rmtree(staging_dir)


PACKAGE CREATED AND ZIP INTEGRITY VERIFIED
Location: C:\Users\ADEGOKE\Desktop\SWATS\SWATS_Kaggle_Transfer_20261024_225448.zip
Size: 231.59 MB
Files packaged: 200
Original project files were not modified.
